# 08 · Module und Pakete

**Lernziele**

- Du weißt, was Module und Pakete sind und wie ein Projekt aufgebaut ist.
- Du kennst die Varianten von `import` und wann du welche nimmst.
- Du verstehst, was bei `import` wirklich passiert.
- Du nutzt `if __name__ == "__main__":` für Skripte.
- Du installierst Pakete von Drittanbietern mit uv.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 2 und 6.

## Module, Pakete, Bibliotheken

| Begriff | Was es ist | Beispiel |
| --- | --- | --- |
| Modul | eine `.py`-Datei | `greetings.py` |
| Paket | ein Ordner mit Modulen (und meist einer `__init__.py`) | `shop/` |
| Standardbibliothek | Module, die mit Python kommen | `math`, `random`, `pathlib`, `json` |
| Drittanbieter-Paket | von [PyPI](https://pypi.org) installiert | `requests`, `pandas` |

Für dieses Notebook liegen Beispiele im Ordner `beispiele/`:

```
beispiele/
├── greetings.py
└── shop/
    ├── __init__.py
    ├── pricing.py
    └── invoice.py
```

Damit Python den Ordner findet, fügen wir ihn zum Suchpfad hinzu. Was das bedeutet, klären wir gleich.

In [ ]:
import sys

sys.path.insert(0, "../beispiele")

## Import-Varianten

In [ ]:
import math                       # ganzes Modul, Zugriff mit math.sqrt
from random import randint        # einzelner Name direkt
import statistics as stats        # ganzes Modul unter kürzerem Namen

print(math.sqrt(16))
print(randint(1, 6))
print(stats.mean([2, 4, 9]))

| Variante | Wann |
| --- | --- |
| `import math` | Standard. Beim Lesen ist klar, woher `math.sqrt` kommt. |
| `from pathlib import Path` | einzelne, oft genutzte Namen |
| `import pandas as pd` | etablierte Abkürzungen |
| `from math import *` | **nie**: holt unkontrolliert alle Namen und überschreibt eigene |

Imports stehen immer **oben** in der Datei. *Explicit is better than implicit:* Beim Lesen soll sofort klar sein, woher jeder Name kommt.

**Kurzübung 1 (Umzugskartons):** 47 Ordner kommen in Umzugskartons, in einen Karton passen 12 Ordner. Berechne die Anzahl der Kartons mit der Funktion `ceil` (aufrunden) aus dem Modul `math`: einmal nach `import math`, einmal nach `from math import ceil`. Welche Fassung nimmst du, und warum?

- **Untersuchen:** Warum ist `47 // 12 + 1` keine gute Lösung? Teste mit 48 Ordnern.

In [ ]:
# Deine Lösung


## Was bei `import` passiert

In [ ]:
import greetings

Die Ausgabe zeigt: Python hat die Datei **ausgeführt**, von oben nach unten. Im Detail passiert Folgendes:

1. Python schaut in `sys.modules`, ob das Modul schon geladen ist. Wenn ja, nimmt es das fertige Modul, fertig.
2. Sonst sucht Python eine passende Datei in den Ordnern aus `sys.path`, der Reihe nach.
3. Die Datei wird einmal ausgeführt. Alle Namen darin landen in einem **Modulobjekt**.
4. Das Modulobjekt wird an den Namen `greetings` gebunden.

Ein zweiter Import führt die Datei deshalb **nicht** erneut aus:

In [ ]:
import greetings

print("greetings" in sys.modules)

Auch ein Modul ist ein Objekt mit Attributen:

In [ ]:
print(type(greetings))
print(greetings.__file__)
print(greetings.DEFAULT_GREETING)
print(greetings.greet("Ada"))

> **Im Notebook:** Änderst du eine Moduldatei, sieht das Notebook die Änderung erst nach einem Kernel-Neustart (oder `importlib.reload(greetings)`).

### Wo sucht Python?

In [ ]:
for folder in sys.path:
    print(repr(folder))  # repr zeigt auch den leeren Eintrag ''

Ganz vorne steht `'../beispiele'`, das wir am Anfang eingefügt haben. Der leere Eintrag `''` steht für den aktuellen Ordner, hier den Ordner des Notebooks. Im Notebook kommt er erst **nach** der Standardbibliothek. Startest du dagegen ein Skript mit `uv run python skript.py`, steht der Ordner des Skripts **ganz vorne**. Das hat eine fiese Folge:

> **Typischer Fehler:** Nenne eine eigene Datei nie wie ein Modul der Standardbibliothek, also nicht `random.py`, `statistics.py`, `csv.py` oder `email.py`. Startest du ein Skript aus demselben Ordner, findet `import random` deine Datei statt der echten, und plötzlich fehlt `random.randint`. Im Notebook fällt das nicht auf, weil dort die Standardbibliothek zuerst kommt.

Gefahrlos ausprobieren lässt sich das mit dem Modul `this` aus Notebook 00, das den Zen of Python ausgibt. Eine Zelle, die mit `%%writefile name.py` beginnt, schreibt ihren Inhalt in diese Datei. Die nächsten beiden Zellen legen so eine eigene `this.py` an und ein Skript `zen.py`, das nur `import this` enthält:

In [ ]:
%%writefile this.py
print("Eigene this.py geladen, nicht der Zen of Python!")

In [ ]:
%%writefile zen.py
import this

Im Notebook liefert `import this` weiter den echten Zen of Python, denn hier kommt die Standardbibliothek zuerst:

In [ ]:
import this

Als Skript sieht das anders aus. Ein `!` am Anfang einer Zeile schickt den Rest der Zeile an die Kommandozeile (das Terminal), statt ihn als Python auszuführen. So startest du `zen.py` als eigenes Programm, und dort steht der Ordner des Skripts ganz vorne:

In [ ]:
!uv run python zen.py

Statt des Zen of Python erscheint die Zeile aus der eigenen `this.py`: Das Skript hat die falsche Datei importiert.

![Im Notebook findet import this zuerst die Standardbibliothek, die eigene this.py im Ordner des Notebooks wird nie erreicht; als Skript steht der Ordner des Skripts vorne, und die eigene this.py wird zuerst gefunden.](../assets/08_suchpfad.png)

**Kurzübung 2 (Zweiter Import):** Sage vorher, welche Ausgabe die folgende Zelle erzeugt, und begründe: Erscheint die Zeile „Modul greetings wird geladen“? Führe die Zelle danach aus.

In [ ]:
import greetings as gr

print(gr is greetings)
print(gr.greet("Grace", "Moin"))

In [ ]:
# Deine Lösung


## Pakete

Ein Paket gruppiert zusammengehörige Module. Importiert wird mit Punkten:

![In from shop.pricing import gross_price, VAT_RATE steht shop für den Ordner shop/ in beispiele/, pricing für die Datei pricing.py darin, und gross_price und VAT_RATE für Namen aus dieser Datei.](../assets/08_paket_import.png)

In [ ]:
from shop.pricing import gross_price, VAT_RATE
from shop.invoice import invoice_line

print(VAT_RATE, gross_price(100))
print(invoice_line("Kaffee", 4.20))

`shop/invoice.py` nutzt selbst `from shop.pricing import gross_price`. Module dürfen sich also gegenseitig importieren.

Ein typisches kleines Projekt sieht so aus:

```
mein_projekt/
├── pyproject.toml      # Projektbeschreibung und Abhängigkeiten (uv)
├── README.md
├── src/
│   └── mein_projekt/
│       ├── __init__.py
│       ├── models.py
│       └── reports.py
└── tests/
```

Datei- und Ordnernamen: `snake_case`, keine Leerzeichen, keine Bindestriche. `mein-projekt.py` lässt sich nicht importieren.

**Kurzübung 3 (Kassenbon aus dem Paket `shop`):** Gib für den Warenkorb `cart` (Nettopreise) mit `invoice_line` je eine Bonzeile aus und berechne die Bruttosumme mit `gross_price`. Erwartet: 19.17 €.

In [ ]:
cart = {"Kaffee": 4.20, "Tasse": 8.40, "Notizblock": 3.50}

In [ ]:
# Deine Lösung


## Skripte: `if __name__ == "__main__":`

Jedes Modul hat einen Namen in `__name__`. Wird die Datei **direkt gestartet**, heißt sie `"__main__"`, beim **Import** trägt sie ihren Modulnamen:

In [ ]:
print(greetings.__name__)
print(__name__)  # das Notebook selbst

Darum steht der Startcode eines Skripts in diesem Block. Er läuft beim Starten, aber nicht beim Import:

```
def main():
    print(greet("Welt"))


if __name__ == "__main__":
    main()
```

```
uv run python beispiele/greetings.py
```

**Kurzübung 4 (Modul als Skript starten):** Sage vorher, welche Zeilen erscheinen, wenn du `greetings.py` als Skript startest, und vergleiche mit der Ausgabe von `import greetings` weiter oben. Starte die Datei dann im Notebook mit `!uv run python ../beispiele/greetings.py` (das `!` schickt die Zeile an die Kommandozeile, siehe „Wo sucht Python?“).

In [ ]:
# Deine Lösung


## Pakete von Drittanbietern

Pakete von PyPI installierst du mit uv. Sie landen in der `.venv` des Projekts und werden in `pyproject.toml` eingetragen. Die `.venv` ist ein Ordner im Projekt mit einer eigenen Python-Umgebung: uv legt sie an, und installierte Pakete gelten nur für dieses Projekt, nicht für den ganzen Rechner.

```
uv add requests
```

Danach funktioniert `import requests` wie bei jedem anderen Modul. Prüfe vorher kurz, ob ein Paket gepflegt und verbreitet ist (Downloads, letzte Version, Dokumentation).

> **Achtung bei Paketnamen von einer KI:** KIs erfinden manchmal Paketnamen, die es nicht gibt; Angreifer registrieren solche Namen und legen Schadcode hinein. Suche einen vorgeschlagenen Namen vor `uv add` auf [pypi.org](https://pypi.org) und prüfe, ob es das gemeinte, verbreitete Paket ist.

## Zusammenfassung

- Modul = `.py`-Datei, Paket = Ordner mit Modulen.
- Bevorzugt `import modul` oder `from modul import name`, nie `import *`.
- `import` sucht in `sys.path`, führt die Datei **einmal** aus und merkt sie sich in `sys.modules`.
- Eigene Dateien nie wie Standardmodule nennen: Ein Skript findet sonst die eigene Datei zuerst.
- Startcode in `if __name__ == "__main__":`; im Notebook startet `!uv run python datei.py` ein Skript.
- Drittanbieter-Pakete: `uv add paketname`, installiert in die `.venv` des Projekts. Paketnamen von einer KI vorher auf pypi.org prüfen.

> **Weiter im Buch:** Als Nächstes käme Kapitel 7 (Dateien). Wir ziehen Kapitel 9 (Fehlerbehandlung) vor, weil wir `try` und `with` beim Arbeiten mit Dateien brauchen.